## Import libraries

In [ ]:
pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 10.1 MB/s eta 0:00:00


In [ ]:
#from bs4 import BeautifulSoup
#import urllib.request
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import itertools
import pandas as pd
import pickle
from IPython.display import display, clear_output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
paper_x = pd.read_excel('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/detik_hukum_50_artikel.xlsx')

paper_x = paper_x.tail(3)

paper = paper_x.values.tolist()



print(paper)

[['https://news.detik.com/berita/d-8384391/sidang-vonis-delpedro-dkk-di-kasus-penghasutan-digelar-jumat-lusa', 'Sidang Vonis Delpedro dkk di Kasus Penghasutan Digelar Jumat Lusa', 'Rabu, 04 Mar 2026 19:51 WIB', 'Sidang kasus penghasutan terkaitdemonstrasi berujung kericuhan pada Agustuslalu dengan terdakwa Direktur Eksekutif LokataruDelpedro Marhaendkk memasuki babak akhir. Sidang vonis Delpedro bersama tiga terdakwa lainnya akan digelar pada Jumat (6/3) lusa. "Selanjutnya putusan akan dibacakan pada Jumat ya," ujar ketua majelis hakim Harika Nova Yeri di Pengadilan Negeri Jakarta Pusat (PN Jakpus), Rabu (4/3/2026). Tiga terdakwa lainnya dalam perkara ini yaitu admin @gejayanmemanggil Syahdan Husein, staf Lokataru Foundation Muzaffar, serta mahasiswa Universitas Riau Khariq Anhar. Hakim mengatakan sidang vonis akan digelar sekitar pukul 14.00 WIB. SCROLL TO CONTINUE WITH CONTENT "Sidang dibuka lagi habis Jumatan ya, jam dua," ujar hakim. Sebelumnya, PN Jakpus menggelar sidang tuntutan 

In [ ]:
print("Saving data to pickle/detik_hukum_50_artikel.pkl..")

# Check if 'paper' is defined and not empty before attempting to pickle
if 'paper' in locals() and paper and len(paper) > 0:
    with open(
        '/content/drive/MyDrive/Kelompok 6 sistem temu kembali/detik_hukum_50_artikel.pkl', # Corrected file extension to .pkl
        'wb'
    ) as f:
        pickle.dump(paper, f)
    print("Success.")
else:
    print("Skipping saving to pickle: 'paper' data was not loaded successfully or is empty.")

Saving data to pickle/detik_hukum_50_artikel.pkl..
Success.


## Preprocess the documents

In [ ]:
# preprocessing

factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()
words = []
processed_paper = []
for x in tqdm(paper, desc='paper', unit='paper'):
    text = x[3]
    text = text.lower()

    remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
    text = text.translate(remove_punctuation_map)
    text = stopword.remove(text)
    text = text.split()
    text = [stemmer.stem(x) for x in text]
    processed_paper.append(' '.join(text))
    text = list(set(text))
    words += text

   # print (text)

paper: 100%|██████████| 3/3 [00:18<00:00,  6.31s/paper]


In [ ]:
# save results to 'corpus/processed_paper.xlsx'

print("Saving data to corpus/processed_detik_hukum.xlsx..")
df = pd.DataFrame(processed_paper)
df.to_excel('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/Corpus/processed_detik_hukum.xlsx', header=False, index=False)
print("Success.")

Saving data to corpus/processed_detik_hukum.xlsx..
Success.


In [ ]:
# save results to 'pickle/processed_paper.pkl'

print("Saving data to pickle/processed_paper.pkl..")
with open('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/Pickle/processed_detik_hukum.pkl', 'wb') as f:
    pickle.dump(processed_paper, f)
print("Success.")

Saving data to pickle/processed_paper.pkl..
Success.


In [ ]:
# save words to 'corpus_detik_hukum.xlsx'

print("Saving data to corpus_detik_hukum.xlsx..")
df = pd.DataFrame(words)
df.to_excel('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/Corpus/corpus_detik_hukum.xlsx', header=False, index=False)
print("Success.")

Saving data to corpus_detik_hukum.xlsx..
Success.


In [ ]:
# save words to 'pickle/words.pkl'

print("Saving data to pickle/detik_hukum_pickle..")
with open('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/Pickle/detik_hukum_pickle.pkl', 'wb') as f:
    pickle.dump(words, f)
print("Success.")

Saving data to pickle/detik_hukum_pickle..
Success.


## Generate thesaurus

In [ ]:
# scrap from sinonimkata.com

thesaurus = {}
words = list(set(words))
for x in tqdm(words, desc='word', unit='word'):
    name = x
    data = { "q": name }
    encoded_data = urllib.parse.urlencode(data).encode("utf-8")
    content = urllib.request.urlopen("http://www.sinonimkata.com/search.php", encoded_data)
    soup = BeautifulSoup(content, 'html.parser')
    try:
        synonym = soup.find('td', attrs={'width': '90%'}).find_all('a')
        synonym = [x.getText() for x in synonym]
        thesaurus[x] = [x] + synonym
    except:
        thesaurus[x] = [name]

word: 100%|██████████| 356/356 [05:11<00:00,  1.14word/s]


In [ ]:
# save results to 'corpus/thesaurus.xlsx'

thesaurus_list = [[x, thesaurus[x]] for x in thesaurus]
print("Saving data to corpus/thesaurus-juz30.xlsx..")
df = pd.DataFrame(thesaurus_list)
df.to_excel('/content/drive/MyDrive/Colab Notebooks/SE Quran/Jupyter Notebook/corpus/thesaurus-juz30.xlsx', header=False, index=False)
print("Success.")

Saving data to corpus/thesaurus-juz30.xlsx..
Success.


In [ ]:
# save results to 'pickle/thesaurus.pkl'

print("Saving data to pickle/juz30.pkl..")
with open('/content/drive/MyDrive/Colab Notebooks/SE Quran/Jupyter Notebook/pickle/juz30.pkl', 'wb') as f:
    pickle.dump(thesaurus, f)
print("Success.")

Saving data to pickle/juz30.pkl..
Success.


## Testing

### Test 1. Query: 'pengembangan aplikasi'

In [ ]:
# insert query here
init_query = 'Hakim'

#### Without query expansion:

In [ ]:
# build tf_idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]
print("Query used: " +' '.join(query))

Query used: hakim


In [ ]:
# process the query

max_result = []
x = [' '.join(query)]
paper_tfidf = vectorizer.fit_transform(x + processed_paper)
q = paper_tfidf[0]
result = cosine_similarity(paper_tfidf, q)
idx = np.argsort(-result,axis=0).flatten()
final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 10 [document, scores, query]:")
for x in new_result[1:5
]:
    print(x)

Number of documents returned: 3.
Top 10 [document, scores, query]:
[2, np.float64(0.24154814730040866), ['hakim']]
[3, np.float64(0.09813092665299217), ['hakim']]
[1, np.float64(0.06639008358991391), ['hakim']]


In [ ]:
for x in new_result[1:5]:
    print('Result', x[0])
    print('QUERY', x[2])

    print('Judul : ', paper[x[0]-1][2])
    print('Isi', paper[x[0]-1][3])
    print(paper[x[0]-1][3])

    print()

Result 2
QUERY ['hakim']
Judul :  Selasa, 03 Mar 2026 21:58 WIB
Isi Terdakwa pengacara Junaedi Saibih divonis bebas dalam kasus suap hakim pemvonis lepas perkara minyak goreng (migor). Hakim menyatakan jaksa penuntut umum (JPU) gagal membuktikan perbuatan Junaedi dalam perkara suap. "Mengadili, menyatakan Terdakwa Junaedi Saibih tersebut di atas, tidak terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana sebagaimana didakwakan dalam dakwaan alternatif kesatu, alternatif kedua, dan alternatif ketiga penuntut umum," ujar ketua majelis hakim Efendi saat membacakan amar putusan Junaedi Saibih di Pengadilan Tipikor Jakarta Pusat, Selasa (3/3/2026). "Membebaskan Terdakwa oleh karena itu dari seluruh dakwaan Penuntut Umum," imbuh hakim. SCROLL TO CONTINUE WITH CONTENT Hakim menyatakan Junaedi tidak pernah terbang ke Singapura untuk rapat langsung dengan Wilmar Group Singapura selaku pihak prinsipal yang juga terlihat dari alat bukti paspor Junaedi. Hakim menyatakan jaksa gagal 

In [ ]:
# save results to 'result/'
file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Judul: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_original.xlsx..")
df = pd.DataFrame(file)

df.to_excel('/content/drive/MyDrive/Kelompok 6 sistem temu kembali/result/' +init_query+ '_original.xlsx', header=False, index=False)
print("Success.")

Saving result to result/Hakim_original.xlsx..
Success.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ===== UI =====
title = widgets.HTML("<h2 style='color:#333;'>🔎 Pencarian Artikel</h2>")

search_box = widgets.Text(
    placeholder='Ketik kata kunci...',
    description='Search:',
    layout=widgets.Layout(width='60%')
)

search_button = widgets.Button(
    description="Search",
    button_style='primary',
    icon='search'
)

output = widgets.Output()

# ===== FUNGSI VALIDASI LINK =====
def fix_link(raw_link, judul):
    link = str(raw_link).strip()

    # Jika kosong
    if link == "" or link == "-" or link.lower() == "nan":
        return "https://www.google.com/search?q=" + judul.replace(" ", "+")

    # Jika sudah URL lengkap
    if link.startswith("http"):
        return link

    # Jika hanya domain tanpa http
    if "detik.com" in link:
        return "https://" + link

    # Jika path saja (/berita/xxx)
    if link.startswith("/"):
        return "https://news.detik.com" + link

    # Fallback ke Google
    return "https://www.google.com/search?q=" + judul.replace(" ", "+")


# ===== FUNGSI SEARCH =====
def on_search(b):
    query_input = search_box.value
    output.clear_output()

    if query_input.strip() == "":
        with output:
            print("⚠️ Masukkan kata kunci dulu!")
        return

    # ===== PREPROCESSING =====
    query = query_input.lower()
    query = query.translate(remove_punctuation_map)
    query = query.split()

    x_query = [' '.join(query)]

    # ❗ FIX: jangan fit ulang tiap search (lebih stabil)
    paper_tfidf = vectorizer.transform(x_query + processed_paper)

    q = paper_tfidf[0]
    result = cosine_similarity(paper_tfidf, q)

    # ===== FILTER & SORT =====
    final = []
    for num, y in enumerate(result):
        if y[0] > 0:
            final.append([num, y[0]])

    max_result = sorted(final, key=lambda x: x[1], reverse=True)

    # ===== OUTPUT =====
    with output:
        total = max(0, len(max_result) - 1)
        print(f"\nMenampilkan {total} hasil untuk: '{query_input}'\n")

        if total == 0:
            print("❌ Tidak ada hasil ditemukan.")
            return

        for x in max_result[1:6]:
            idx = x[0] - 1

            # Safety check index
            if idx < 0 or idx >= len(paper):
                continue

            data = paper[idx]

            # ===== AMBIL DATA =====
            judul = data[0] if len(data) > 0 else "-"
            tanggal = data[1] if len(data) > 1 else "-"
            isi = data[2] if len(data) > 2 else "-"
            raw_link = data[3] if len(data) > 3 else ""

            # ===== FIX LINK =====
            link = fix_link(raw_link, judul)

            # DEBUG (optional, bisa dihapus nanti)
            print("DEBUG LINK:", raw_link, "->", link)

            # ===== TAMPILAN CARD =====
            display(HTML(f"""
            <div style="
                border:1px solid #ddd;
                border-radius:10px;
                padding:15px;
                margin-bottom:15px;
                box-shadow:0 2px 5px rgba(0,0,0,0.1);
            ">
                <h3 style="margin:0;">{judul}</h3>
                <p style="color:gray; font-size:13px;">📅 {tanggal}</p>
                <p style="color:#444;">{str(isi)[:150]}...</p>

                <a href="{link}" target="_blank" style="color:#1a73e8; text-decoration:none;">
                    🔗 Baca artikel lengkap
                </a>
            </div>
            """))


# ===== EVENT =====
search_button.on_click(on_search)

# ===== LAYOUT =====
ui = widgets.VBox([
    title,
    widgets.HBox([search_box, search_button]),
    output
])

display(ui)

DEBUG LINK: Sidang kasus penghasutan terkaitdemonstrasi berujung kericuhan pada Agustuslalu dengan terdakwa Direktur Eksekutif LokataruDelpedro Marhaendkk memasuki babak akhir. Sidang vonis Delpedro bersama tiga terdakwa lainnya akan digelar pada Jumat (6/3) lusa. "Selanjutnya putusan akan dibacakan pada Jumat ya," ujar ketua majelis hakim Harika Nova Yeri di Pengadilan Negeri Jakarta Pusat (PN Jakpus), Rabu (4/3/2026). Tiga terdakwa lainnya dalam perkara ini yaitu admin @gejayanmemanggil Syahdan Husein, staf Lokataru Foundation Muzaffar, serta mahasiswa Universitas Riau Khariq Anhar. Hakim mengatakan sidang vonis akan digelar sekitar pukul 14.00 WIB. SCROLL TO CONTINUE WITH CONTENT "Sidang dibuka lagi habis Jumatan ya, jam dua," ujar hakim. Sebelumnya, PN Jakpus menggelar sidang tuntutan terkait demonstrasi berujung kericuhan pada Agustus tahun lalu dengan terdakwa Direktur Eksekutif Lokataru Delpedro Marhaen dan tiga orang rekannya. Delpedro dkk dituntut dua tahun penjara. "Yang pert

#### With query expansion:

In [ ]:
# build tf-idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]

In [ ]:
# generate query expansion

product_query = []
list_synonym = []
for x in query:
    if x in words:
        list_synonym.append(thesaurus[x])
    else:
        name = x
        data = { "q": name }
        encoded_data = urllib.parse.urlencode(data).encode("utf-8")
        content = urllib.request.urlopen("http://www.sinonimkata.com/search.php", encoded_data)
        soup = BeautifulSoup(content, 'html.parser')
        try:
            synonym = soup.find('td', attrs={'width': '90%'}).find_all('a')
            synonym = [x.getText() for x in synonym]
            thesaurus[x] = [x] + synonym
            list_synonym.append(thesaurus[x])
        except:
            list_synonym.append([x])
qs = []
for x in itertools.product(*list_synonym):
    x = [stemmer.stem(y) for y in x]
    qs.append([' '.join(x)])
print("Queries used:")
for x in qs:
    print("-", x[0])

NameError: name 'thesaurus' is not defined

In [ ]:
# process the query

max_result = []
for x in qs:
    paper_tfidf = vectorizer.fit_transform(x + processed_paper)
    q = paper_tfidf[0]
    result = cosine_similarity(paper_tfidf, q)
    idx = np.argsort(-result,axis=0).flatten()
    final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
    max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 10 [document, scores, query]:")
for x in new_result[1:10]:
    print(x)

NameError: name 'qs' is not defined

In [ ]:
# show top 5 results

for x in new_result[1:20]:
    print('Result', x[0])
    print('QUERY', x[2])

    print('Surah : ', paper[x[0]-1][2])
    print('Ayat', paper[x[0]-1][3])
    print(paper[x[0]-1][4])

    print()

Result 11
QUERY ['pidana']
Surah :  Jumat, 06 Mar 2026 11:42 WIB
Ayat Direktur Eksekutif LokataruDelpedro Marhaendan staf Lokataru, Muzaffar Salim, mengajukan gugatan terhadap sejumlah pasal di UU 1/2023 tentang KUHP. Pasal yang digugat itu antara lain pasal penghasutan dan penyebaran hoaks. Dilihat dari situs resmi MK, Jumat (6/3/2026), gugatan tersebut teregistrasi dengan nomor 93/PUU-XXIV/2026. Mereka mengajukan gugatan terhadap Pasal 246, Pasal 264 ayat (1) dan ayat (2), serta pasal 264. Berikut isi pasal-pasal yang digugat: SCROLL TO CONTINUE WITH CONTENT Pasal 246 Dipidana dengan pidana penjara paling lama 4 tahun atau pidana denda paling banyak kategori V (Rp 500 juta), setiap Orang yang Di Muka Umum dengan lisan atau tulisan: a. menghasut orang untuk melakukan Tindak Pidana; ataub. menghasut orang untuk melawan penguasa umum dengan Kekerasan. Pasal 263 (1) Setiap Orang yang menyiarkan atau menyebarluaskan berita atau pemberitahuan padahal diketahuinya bahwa berita atau pemberit

IndexError: list index out of range

In [ ]:
# save results to 'result/'

file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Title: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_expansion.xlsx..")
df = pd.DataFrame(file)
df.to_excel('result/' +init_query+ '_expansion.xlsx', header=False, index=False)
print("Success.")

### Test 2. Query: 'pengolahan dokumen'

In [ ]:
# insert query here

init_query = 'pengolahan dokumen'

#### Without query expansion:

In [ ]:
# build tf_idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]
print("Query used: " +' '.join(query))

In [ ]:
# process the query

max_result = []
x = [' '.join(query)]
paper_tfidf = vectorizer.fit_transform(x + processed_paper)
q = paper_tfidf[0]
result = cosine_similarity(paper_tfidf, q)
idx = np.argsort(-result,axis=0).flatten()
final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 5 [document, scores, query]:")
for x in new_result[1:6]:
    print(x)

In [ ]:
# show top 5 results

for x in new_result[1:6]:
    print('Result', x[0])
    print('QUERY', x[2])
    print(paper[x[0]-1][1])
    print(paper[x[0]-1][2][:200] + '...')
    print()

In [ ]:
# save results to 'result/'

file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Title: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_original.xlsx..")
df = pd.DataFrame(file)
df.to_excel('result/' +init_query+ '_original.xlsx', header=False, index=False)
print("Success.")

#### With query expansion:

In [ ]:
# build tf-idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]

In [ ]:
# generate query expansion

product_query = []
list_synonym = []
for x in query:
    if x in words:
        list_synonym.append(thesaurus[x])
    else:
        name = x
        data = { "q": name }
        encoded_data = urllib.parse.urlencode(data).encode("utf-8")
        content = urllib.request.urlopen("http://www.sinonimkata.com/search.php", encoded_data)
        soup = BeautifulSoup(content, 'html.parser')
        try:
            synonym = soup.find('td', attrs={'width': '90%'}).find_all('a')
            synonym = [x.getText() for x in synonym]
            thesaurus[x] = [x] + synonym
            list_synonym.append(thesaurus[x])
        except:
            list_synonym.append([x])
qs = []
for x in itertools.product(*list_synonym):
    x = [stemmer.stem(y) for y in x]
    qs.append([' '.join(x)])
print("Queries used:")
for x in qs:
    print("-", x[0])

In [ ]:
# process the query

max_result = []
for x in qs:
    paper_tfidf = vectorizer.fit_transform(x + processed_paper)
    q = paper_tfidf[0]
    result = cosine_similarity(paper_tfidf, q)
    idx = np.argsort(-result,axis=0).flatten()
    final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
    max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 5 [document, scores, query]:")
for x in new_result[1:6]:
    print(x)

In [ ]:
# show top 5 results

for x in new_result[1:6]:
    print('Result', x[0])
    print('QUERY', x[2])
    print(paper[x[0]-1][1])
    print(paper[x[0]-1][2][:200] + '...')
    print()

In [ ]:
# save results to 'result/'

file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Title: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_expansion.xlsx..")
df = pd.DataFrame(file)
df.to_excel('result/' +init_query+ '_expansion.xlsx', header=False, index=False)
print("Success.")

### Test 3. Query: 'deteksi kendaraan'

In [ ]:
# insert query here

init_query = 'deteksi kendaraan'

#### Without query expansion:

In [ ]:
# build tf_idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]
print("Query used: " +' '.join(query))

In [ ]:
# process the query

max_result = []
x = [' '.join(query)]
paper_tfidf = vectorizer.fit_transform(x + processed_paper)
q = paper_tfidf[0]
result = cosine_similarity(paper_tfidf, q)
idx = np.argsort(-result,axis=0).flatten()
final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 5 [document, scores, query]:")
for x in new_result[1:6]:
    print(x)

In [ ]:
# show top 5 results

for x in new_result[1:6]:
    print('Result', x[0])
    print('QUERY', x[2])
    print(paper[x[0]-1][1])
    print(paper[x[0]-1][2][:200] + '...')
    print()

In [ ]:
# save results to 'result/'

file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Title: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_original.xlsx..")
df = pd.DataFrame(file)
df.to_excel('result/' +init_query+ '_original.xlsx', header=False, index=False)
print("Success.")

#### With query expansion:

In [ ]:
# build tf-idf

vectorizer = TfidfVectorizer(use_idf=True)
query = init_query
query = query.lower()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)
query = query.translate(remove_punctuation_map)
query = stopword.remove(query)
query = query.split()
query = [stemmer.stem(x) for x in query]

In [ ]:
# generate query expansion

product_query = []
list_synonym = []
for x in query:
    if x in words:
        list_synonym.append(thesaurus[x])
    else:
        name = x
        data = { "q": name }
        encoded_data = urllib.parse.urlencode(data).encode("utf-8")
        content = urllib.request.urlopen("http://www.sinonimkata.com/search.php", encoded_data)
        soup = BeautifulSoup(content, 'html.parser')
        try:
            synonym = soup.find('td', attrs={'width': '90%'}).find_all('a')
            synonym = [x.getText() for x in synonym]
            thesaurus[x] = [x] + synonym
            list_synonym.append(thesaurus[x])
        except:
            list_synonym.append([x])
qs = []
for x in itertools.product(*list_synonym):
    x = [stemmer.stem(y) for y in x]
    qs.append([' '.join(x)])
print("Queries used:")
for x in qs:
    print("-", x[0])

In [ ]:
# process the query

max_result = []
for x in qs:
    paper_tfidf = vectorizer.fit_transform(x + processed_paper)
    q = paper_tfidf[0]
    result = cosine_similarity(paper_tfidf, q)
    idx = np.argsort(-result,axis=0).flatten()
    final = [[num, y[0], x] for num, y in enumerate(result) if y[0] > 0.0]
    max_result += final
max_result = sorted(max_result, key=lambda x: x[1], reverse=True)
set_result = set()
new_result = []
for item in max_result:
    if item[0] not in set_result:
        set_result.add(item[0])
        new_result.append(item)
    else:
        pass
print("Number of documents returned: " +str(len(new_result)-1)+ ".")
print("Top 5 [document, scores, query]:")
for x in new_result[1:6]:
    print(x)

In [ ]:
# show top 5 results

for x in new_result[1:6]:
    print('Result', x[0])
    print('QUERY', x[2])
    print(paper[x[0]-1][1])
    print(paper[x[0]-1][2][:200] + '...')
    print()

In [ ]:
# save results to 'result/'

file = []
for x in new_result[1:]:
    temp = []
    temp.append('Document: ' +str(x[0]))
    temp.append('Query: ' +x[2][0])
    temp.append('Title: ' +paper[x[0]-1][1])
    temp.append(paper[x[0]-1][2])
    file.append(temp)

print("Saving result to result/" +init_query+ "_expansion.xlsx..")
df = pd.DataFrame(file)
df.to_excel('/content/drive/MyDrive/Colab Notebooks/SE Quran/Jupyter Notebook/result/' +init_query+ '_expansion.xlsx', header=False, index=False)
print("Success.")